# **Exploring the Housing Market in Ames, Iowa**

## **Goal**

1. How have houses changed in their features over the years?

2. How have market dynamics shifted between properties designed for single households, such as single‑family homes and townhouses, and those designed for multiple households or shared living, such as duplexes and two‑family conversions, over time?

3. Which features make houses sell best?

## **Introduction**

The housing market affects everyone, from big real estate investors to families looking for a home. It's a complicated system, with prices and trends shifting constantly over time. In this report, we aim to make sense of these changes and provide a clearer picture of how the market has evolved. To do this, we'll analyze data from 1,460 property sales recorded between 2006 and 2010. Within this dataset, our focus will be on....


The data was collected by De Cock and recorded 1460 properties in Ames, IA.

Brief description of columns:

**1. Identification**
- **Id:** Property ID (unique identifier).

__2. Sale Information__
- **SalePrice:** Target variable — final selling price.
- **MoSold:** Month sold.
- **YrSold:** Year sold.

__3. Lot & Land__
- **MSSubClass:** Dwelling type (e.g., 1-story, 2-story).
- **MSZoning:** Zoning classification.
- **LotArea:** Lot size (sq ft).

__4. Building & Construction__

- **BldgType:** Building type (Single-family, Duplex).
- **HouseStyle:** House style (1-story, 2-story, split-level).
- **YearBuilt:** Year built.
- **Exterior1st/Exterior2nd:** Exterior covering materials.
- **MasVnrType:** Masonry veneer type.
- **ExterQual:** Exterior quality.
- **ExterCond:** Exterior condition.

__5. Interior Features__

- **GrLivArea:** Above-ground living area.
- **BsmtFullBath/BsmtHalfBath:** Basement bathrooms.
- **FullBath/HalfBath:** Bathrooms above ground.
- **BedroomAbvGr:** Bedrooms above ground.
- **TotRmsAbvGrd:** Total rooms above ground.



**This dataset represents the sale of properties in Ames, Iowa, where every single record represents a sale of individual house.
There are identification, land lot information, house characteristics, exterior, sales information and other features describing each house. The data of selling houses spans from 2006 to 2010.**

### __Import Libraries__

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
%matplotlib inline
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import sidetable
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import MDS
import statsmodels.api as sm

### __Display Setting__

In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
import warnings
warnings.filterwarnings("ignore")
# set the decimal places to 2
pd.set_option("display.float_format", "{:.2f}".format)

### **Basic Understanding & Data Ingestion**

In [ ]:
path = Path.cwd().parent
data_path = path.joinpath("data", "raw")
files = []
for file in data_path.rglob("*.csv"):
    files.append(file)
    print(file.name)
    print(files.index(file), " ", file)

In [ ]:
# Read train and test
train_data = pd.read_csv(files[2])
test_data = pd.read_csv(files[1])
print("Training Dataset:")
display(train_data.head(1))
print("\n")
print("Testing Dataset:")
display(test_data.head(1))


In [ ]:
# Basic info of datasets
print(f"Training Dataset Shape : {train_data.shape}")
print(f"Testing Dataset Shape : {test_data.shape}")


In [ ]:
# remove whitespace from columns name, replace space to '_' for columns name for conveince

train_data.columns = train_data.columns.str.strip().str.replace(" ", "_").str.replace(".", "_")#.str.lower()
test_data.columns = test_data.columns.str.strip().str.replace(" ", "_").str.replace(".", "_")

In [ ]:
# Selecting columns for analysis in this notebook and define new train and test datasets
cols = ['Id', 'MSSubClass', 'MSZoning', 'LotArea', 'BldgType',
    'HouseStyle', 'YearBuilt', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
    'MasVnrArea', 'ExterQual', 'ExterCond', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
    'HalfBath', 'BedroomAbvGr', 'TotRmsAbvGrd', 'MoSold', 'YrSold', 'SalePrice']
    
df_train = train_data[cols]


In [ ]:
new_cols = [x for x in cols if x != "SalePrice"]
df_test = test_data[new_cols]

### **Columns Formatting and Consistency**

In [ ]:
dtype_df = pd.DataFrame(index=df_train.columns, columns=['Dtype','nunique','unique'])
dtype_df['Dtype'] = df_train.dtypes
dtype_df['nunique'] = df_train.nunique()
dtype_df['unique'] = [df_train[col].unique() for col in df_train.columns]
dtype_df


### **Exploratory Data Analysis with Charts**

Let's explore the questions that were outlined in the goal section:



**`Question 1.` How have houses changed in their features over the years?**

In [ ]:
# Create a new column for decade bins
df_train["DecadeBuilt"] = (df_train['YearBuilt']//10)*10

In [ ]:
num_cols = ['GrLivArea', 'LotArea']
cat_cols = ['BedroomAbvGr', 'TotRmsAbvGrd', 'FullBath', 'HalfBath']

avg_features = (
    df_train.groupby(['DecadeBuilt'])[num_cols].mean()
    )
avg_features.columns = ["avg_"+col.lower() for col in avg_features.columns]
df_features = avg_features.reset_index()

# Reshape the long format
df_vars = df_features.melt(
    id_vars="DecadeBuilt",
    value_vars=['avg_grlivarea','avg_lotarea'],
    var_name="Features", value_name="average_values"
)
df_vars.head()
#fig, ax = plt.subplots(1, 2, figsize=(15, 8))
sns.lmplot(
    data=df_vars, 
    x="DecadeBuilt", y="average_values", 
    hue="Features", palette=sns.color_palette("Set2", 6)
);



In [ ]:
avg_cat = (
    df_train.groupby(['DecadeBuilt'])[cat_cols].mean()
    )
avg_cat.columns = ["avg_"+col.lower() for col in avg_cat.columns]
df_cat = avg_cat.reset_index()
# Reshape the long format
df_dimensions = df_cat.melt(
    id_vars="DecadeBuilt",
    value_vars=['avg_bedroomabvgr','avg_totrmsabvgrd', 'avg_fullbath', 'avg_halfbath'],
    var_name="Features", value_name="average_values"
)
df_dimensions.head()
sns.lmplot(
    data=df_dimensions, 
    x="DecadeBuilt", y="average_values", 
    hue="Features", palette=sns.color_palette("Set2", 4)
);


In [ ]:
df_train.columns

In [ ]:
import statsmodels.formula.api as smf
model = smf.ols("DecadeBuilt ~ LotArea  + MasVnrArea", data=df_train).fit()
print(model.summary())


In [ ]:
import statsmodels.formula.api as smf
model = smf.ols("SalePrice ~ DecadeBuilt+ BedroomAbvGr+ FullBath + HalfBath + TotRmsAbvGrd", data=df_train).fit()
print(model.summary())


In [ ]:
df_train.columns

In [ ]:
df_train.groupby(['DecadeBuilt', 'BldgType']).size().unstack(fill_value=0).plot(kind='barh', colormap="Set2", stacked=True, figsize=(15, 5));

Single‑family homes have been the main type of housing built over time, and its peak during the mid‑20th century. In more recent years, the mix of building types has become more diversified. Townhouses, which were almost nonexistent in earlier decades, have gradually increased as cities grow denser and people look for smaller, easier‑to‑maintain homes. Duplexes have stayed at a low but steady level, while two‑family conversions have declined as newer homes are built to serve those needs directly. Overall, the data shows a shift from a market dominated by single‑family homes to one with a broader range of housing types.

**`Question 2.` How have market dynamics shifted between properties designed for single households, such as single‑family homes and townhouses, and those designed for multiple households or shared living, such as duplexes and two‑family conversions, over time?**

In [ ]:
u = df_train.groupby(['BldgType']).agg(
    avg_rooms = ('TotRmsAbvGrd', 'mean'),
    avg_bed_rooms = ('BedroomAbvGr', 'mean')
).reset_index()
u

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=u.sort_values(by='avg_rooms', ascending=True), x='BldgType', y='avg_rooms', ax=ax[0])
sns.barplot(data=u.sort_values(by='avg_bed_rooms', ascending=True), x='BldgType', y='avg_bed_rooms', ax=ax[1]);

In [ ]:
v = df_train.groupby(['HouseStyle']).agg(
    avg_rooms = ('TotRmsAbvGrd', 'mean'),
    avg_bed_rooms = ('BedroomAbvGr', 'mean')
).reset_index()
v

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=v.sort_values(by="avg_rooms", ascending=False), x='HouseStyle', y='avg_rooms', ax=ax[0])
sns.barplot(data=v.sort_values(by='avg_bed_rooms', ascending=False), x='HouseStyle', y='avg_bed_rooms', ax=ax[1]);

In [ ]:
j= df_train.pivot_table(index='HouseStyle', columns='DecadeBuilt', values='SalePrice', aggfunc='mean').fillna(0)
sns.heatmap(j, cmap='coolwarm');

In [ ]:
k= df_train.query("BldgType.isin(['1Fam', 'Duplex'])").pivot_table(index='BldgType', columns='DecadeBuilt', values='SalePrice', aggfunc='mean').fillna(0)
sns.heatmap(k, cmap='coolwarm');

In [ ]:
k= df_train.query("BldgType.isin(['TwnhsE', 'Twnhs', '2fmCon'])").pivot_table(index='BldgType', columns='DecadeBuilt', values='SalePrice', aggfunc='mean').fillna(0)
sns.heatmap(k, cmap='coolwarm');

In [ ]:
df_train['BldgType'].value_counts()

In [ ]:
df_train.columns

**`Question 3.` Which features make houses sell best?**

In [150]:
df_train.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotArea', 'BldgType', 'HouseStyle',
       'YearBuilt', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea',
       'ExterQual', 'ExterCond', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath',
       'FullBath', 'HalfBath', 'BedroomAbvGr', 'TotRmsAbvGrd', 'MoSold',
       'YrSold', 'SalePrice', 'DecadeBuilt'],
      dtype='object')

In [153]:
cols = ['YearBuilt', 'LotArea', 'GrLivArea', 'BldgType', 
        'HouseStyle']
# Create log-transformed target
df_train['LogSalePrice'] = np.log(df_train['SalePrice'])+1

# Define formula
formula = "LogSalePrice ~ YearBuilt + LotArea + GrLivArea + C(BldgType) + C(HouseStyle)"

# Fit OLS regression
model = smf.ols(formula=formula, data=df_train).fit()

# Print summary
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:           LogSalePrice   R-squared:                       0.747
Model:                            OLS   Adj. R-squared:                  0.744
Method:                 Least Squares   F-statistic:                     304.6
Date:                Wed, 25 Mar 2026   Prob (F-statistic):               0.00
Time:                        03:03:26   Log-Likelihood:                 271.74
No. Observations:                1460   AIC:                            -513.5
Df Residuals:                    1445   BIC:                            -434.2
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

In [170]:
# Select features
cols = ['YearBuilt', 'LotArea', 'GrLivArea', 'BldgType']
X = df_train[cols]

# Convert categorical to dummies
X_dummy = pd.get_dummies(X, drop_first=True).astype(int)


# Target
y = df_train['SalePrice']

# Add constant
X_dummy = sm.add_constant(X_dummy)

# Fit model
model = sm.OLS(y, X_dummy).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              SalePrice   R-squared:                       0.684
Model:                            OLS   Adj. R-squared:                  0.682
Method:                 Least Squares   F-statistic:                     448.5
Date:                Wed, 25 Mar 2026   Prob (F-statistic):               0.00
Time:                        03:30:05   Log-Likelihood:                -17704.
No. Observations:                1460   AIC:                         3.542e+04
Df Residuals:                    1452   BIC:                         3.547e+04
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const           -2.058e+06   8.27e+04    -